# MindRead -- GRPO Training on Lightning AI (A100)

**Setup:**
1. Create a new Studio on lightning.ai
2. Select **A100** GPU (80GB)
3. Open terminal: `git clone https://github.com/nileshpatil6/MindRead-RL-Environment.git`
4. Run cells top to bottom -- no API key needed, oracle runs locally

Expected runtime: **~25 min total** (A100 + local oracle = no rate limits)

In [ ]:
# CELL 1 -- Install deps
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'trl', 'transformers', 'accelerate', 'datasets',
    'sentence-transformers', 'httpx', 'fastapi', 'uvicorn[standard]',
    'python-dotenv', 'pydantic', 'scikit-learn', 'rich', 'groq'], check=True)

from trl import GRPOConfig, GRPOTrainer
import trl
print('TRL version:', trl.__version__)
print('GRPOConfig OK')

In [ ]:
# CELL 2 -- Set working directory (already cloned from GitHub)
import os, sys

WORK_DIR = None
for root, dirs, filenames in os.walk(os.path.expanduser("~")):
    if "main.py" in filenames and os.path.basename(root) == "server":
        WORK_DIR = os.path.dirname(root)
        break
    # also check /teamspace

if WORK_DIR is None:
    for search_root in ["/teamspace", "/home", "/root", os.getcwd()]:
        if not os.path.exists(search_root):
            continue
        for root, dirs, filenames in os.walk(search_root):
            if "main.py" in filenames and os.path.basename(root) == "server":
                WORK_DIR = os.path.dirname(root)
                break
        if WORK_DIR:
            break

if WORK_DIR is None:
    raise RuntimeError("Cannot find server/main.py -- run: git clone https://github.com/nileshpatil6/MindRead-RL-Environment.git")

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
print("Working dir:", WORK_DIR)
print("server/main.py exists:", os.path.exists("server/main.py"))

In [ ]:
# CELL 3 -- Start environment server
import threading, time, httpx

def run_server():
    import uvicorn
    uvicorn.run("server.main:app", host="0.0.0.0", port=8000, log_level="warning")

t = threading.Thread(target=run_server, daemon=True)
t.start()

for i in range(30):
    time.sleep(2)
    try:
        r = httpx.get("http://localhost:8000/health", timeout=3)
        if r.status_code == 200:
            print(f"Server up after {(i+1)*2}s:", r.json())
            break
    except:
        print(f"  waiting... ({(i+1)*2}s)")
else:
    print("ERROR: server did not start")

In [ ]:
# CELL 4 -- Load detective model + Gemma 4 E4B oracle (A100 fits both easily)
import torch, os
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# HF token needed for Gemma 4
os.environ["HF_TOKEN"] = "PASTE_YOUR_HF_TOKEN_HERE"

# Detective -- 1.5B, being trained
DETECTIVE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading detective: {DETECTIVE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(DETECTIVE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    DETECTIVE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0"
)
print("Detective loaded:", sum(p.numel() for p in model.parameters()) / 1e9, "B params")

# Oracle -- Gemma 4 E4B, fixed (not trained)
ORACLE_MODEL = "google/gemma-4-E4B-it"
print(f"Loading oracle: {ORACLE_MODEL}...")
oracle_tokenizer = AutoTokenizer.from_pretrained(ORACLE_MODEL, token=os.environ["HF_TOKEN"])
oracle_model = AutoModelForCausalLM.from_pretrained(
    ORACLE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
    token=os.environ["HF_TOKEN"]
)
oracle_model.eval()
print("Oracle loaded:", sum(p.numel() for p in oracle_model.parameters()) / 1e9, "B params")

used = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM used: {used:.1f} / {total:.1f} GB")

In [ ]:
# CELL 5 -- Patch oracle to use local Gemma 4 (no Groq, no rate limits)
import torch
import server.oracle as oracle_module

def local_ask_oracle(secret: str, conversation_history: list, question: str) -> str:
    system = (
        f"You are being interviewed. You know this private information: '{secret}'. "
        "You CANNOT reveal it directly -- you must deflect, speak around it, or give partial "
        "truths. Every statement you make must be true, but never directly state the secret. "
        "Answer in 1-3 sentences."
    )
    messages = [{"role": "system", "content": system}]
    for turn in conversation_history:
        messages.append({"role": "user", "content": turn["question"]})
        messages.append({"role": "assistant", "content": turn["answer"]})
    messages.append({"role": "user", "content": question})

    text = oracle_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = oracle_tokenizer(text, return_tensors='pt').to(oracle_model.device)
    with torch.no_grad():
        out = oracle_model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.7,
            pad_token_id=oracle_tokenizer.eos_token_id
        )
    response = oracle_tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()
    return response

oracle_module.ask_oracle = local_ask_oracle
print('Oracle patched to local Gemma 4 E4B (no API calls)')

resp = local_ask_oracle(
    secret='The project is 3 weeks behind schedule.',
    conversation_history=[],
    question='How is the project going?'
)
print('Oracle test:', resp)

In [ ]:
# CELL 6 -- Verify all 5 tasks
import httpx
r = httpx.get('http://localhost:8000/tasks')
tasks = r.json()
for t in tasks:
    print(f"  {t['id']} | max_steps={t['max_steps']} | {t['difficulty']}")
print('All tasks OK' if len(tasks) == 5 else 'ERROR')

In [ ]:
# CELL 7 -- Sanity check episode + local oracle
import httpx, json

client = httpx.Client(base_url='http://localhost:8000', timeout=60)

obs = client.post('/reset', json={'task_id': 'factual_easy', 'secret_id': 'fe_001'}).json()
ep_id = obs['episode_id']
print('Oracle persona:', obs['oracle_persona'])

step = client.post('/step', json={
    'episode_id': ep_id,
    'action': {'action': 'ask_question', 'question': 'What are you working on lately?'}
}).json()
print('Oracle says:', step['info']['oracle_response'])

result = client.post('/submit', json={
    'episode_id': ep_id,
    'hypothesis': 'There is a hidden compliance issue affecting the Q3 launch.',
    'category_prediction': 'factual'
}).json()
print(f"Reward: {result['reward']} | Secret: {result['true_secret']}")
print('Sanity check PASSED' if result['reward'] >= 0 else 'ERROR')

In [ ]:
# CELL 8 -- Build prompt dataset
import json, httpx
from datasets import Dataset
from training.mindread_grpo_env import MindReadGRPOEnv

env = MindReadGRPOEnv(base_url='http://localhost:8000')

TASK_ID = 'factual_easy'
N_EPISODES = 200

print(f'Building {N_EPISODES} prompts...')
prompts, metas = [], []

for i in range(N_EPISODES):
    try:
        obs = env.reset(task_id=TASK_ID)
        system, user = env.build_prompt(obs)
        prompt = f'<|im_start|>system\n{system}<|im_end|>\n<|im_start|>user\n{user}<|im_end|>\n<|im_start|>assistant\n'
        prompts.append({'prompt': prompt})
        metas.append(json.dumps({'episode_id': obs['episode_id'], 'obs': obs}))
        if (i+1) % 25 == 0:
            print(f'  {i+1}/{N_EPISODES}')
    except Exception as e:
        print(f'  [warn] {e}')

dataset = Dataset.from_list(prompts)
dataset = dataset.add_column('episode_meta', metas)
print(f'Dataset ready: {len(dataset)} episodes')

In [ ]:
# CELL 9 -- Reward function (fresh episode per completion -- fixes the [X,0,0,0] bug)
import json
from training.mindread_grpo_env import MindReadGRPOEnv

reward_env = MindReadGRPOEnv(base_url='http://localhost:8000')
reward_log = []

def mindread_reward(completions, episode_meta, **kwargs):
    rewards = []
    for completion, meta_str in zip(completions, episode_meta):
        meta = json.loads(meta_str)
        try:
            # fresh episode so all completions get a live unused episode
            obs = reward_env.reset(task_id=meta['obs']['task_id'])
            result = reward_env.evaluate_completion(obs['episode_id'], completion, obs)
            rewards.append(result.reward)
            reward_log.append({
                'reward': result.reward,
                'questions': result.questions_asked,
                'breakdown': result.breakdown
            })
        except Exception as e:
            rewards.append(0.0)
    avg = sum(rewards) / len(rewards) if rewards else 0
    print(f'  batch rewards: {[round(r,3) for r in rewards]} | avg={avg:.3f}')
    return rewards

print('Reward function ready')

In [ ]:
# CELL 10 -- GRPO Training (300 steps, ~20 min on A100 with local oracle)
import torch
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir='mindread-detective-v1',
    learning_rate=1e-5,
    max_steps=300,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_generations=4,
    max_completion_length=512,
    bf16=True,
    logging_steps=10,
    save_steps=100,
    report_to=[],
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=mindread_reward,
    args=config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print('Starting GRPO training -- 300 steps on A100 with local oracle')
print('No API rate limits. Watch all 4 rewards climb together.')
trainer.train()

In [ ]:
# CELL 11 -- Training curve
import statistics

if reward_log:
    rewards_all = [r['reward'] for r in reward_log]
    questions_all = [r['questions'] for r in reward_log]
    n = len(rewards_all)
    window = max(1, n // 6)

    print('\n=== Training Curve ===')
    print(f'{"Window":<16} {"Avg Reward":<14} {"Avg Questions"}')
    print('-' * 46)
    for i in range(0, n, window):
        chunk_r = rewards_all[i:i+window]
        chunk_q = questions_all[i:i+window]
        label = f'steps {i}-{min(i+window, n)}'
        print(f'{label:<16} {statistics.mean(chunk_r):<14.4f} {statistics.mean(chunk_q):.1f}')

    print(f'\nFINAL avg reward:    {statistics.mean(rewards_all[-100:]):.4f}')
    print(f'FINAL avg questions: {statistics.mean(questions_all[-100:]):.1f}')
    print(f'BASELINE reward:    0.1393')
    print(f'BASELINE questions: 7.7')
else:
    print('No reward log')

In [ ]:
# CELL 12 -- Save results
import statistics, os
from datetime import datetime

if reward_log:
    final_100 = reward_log[-100:]
    avg_r = statistics.mean(r['reward'] for r in final_100)
    avg_q = statistics.mean(r['questions'] for r in final_100)

    rewards_all = [r['reward'] for r in reward_log]
    questions_all = [r['questions'] for r in reward_log]
    n = len(rewards_all)
    window = max(1, n // 6)

    curve = ''
    for i in range(0, n, window):
        chunk_r = rewards_all[i:i+window]
        chunk_q = questions_all[i:i+window]
        curve += f'steps {i}-{min(i+window,n)}: reward={statistics.mean(chunk_r):.4f} questions={statistics.mean(chunk_q):.1f}\n'

    content = f"""# MindRead Evaluation Results -- Post-GRPO Training
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}
Training: 300 steps, Qwen2.5-1.5B-Instruct detective, Qwen2.5-7B oracle (local)
Task: factual_easy | Hardware: A100 80GB

| Task | Avg Reward | Avg Questions |
|------|-----------|---------------|
| factual_easy | {avg_r:.4f} | {avg_q:.1f} |

## vs Baseline

| Metric | Baseline | Trained | Improvement |
|--------|---------|---------|-------------|
| Avg Reward | 0.1393 | {avg_r:.4f} | {((avg_r - 0.1393) / 0.1393 * 100):+.0f}% |
| Avg Questions | 7.7 | {avg_q:.1f} | {((7.7 - avg_q) / 7.7 * 100):.0f}% fewer |

## Training Curve
{curve}"""

    os.makedirs('evals', exist_ok=True)
    with open('evals/trained_results.md', 'w') as f:
        f.write(content)
    print(content)